# Wisconsin Schools: GCS → Dataform → BigQuery

Raw school records (Parquet in GCS) are loaded into BigQuery by a Dataform pipeline, joined with county polygons from `bigquery-public-data.geo_us_boundaries`, and queried with BigQuery GIS. Runs on a GCP VM; see the README for setup.

## Configuration

In [1]:
import json

with open("config.json") as f:
    config = json.load(f)

PROJECT = config["project"]   # GCP project id
BUCKET = config["bucket"]     # GCS bucket holding wi-schools-raw.parquet
DATASET = config["dataset"]   # BigQuery dataset the Dataform pipeline writes to
REGION = config.get("region", "us-central1")
REPO = config.get("dataform_repository", "wi-schools-pipeline")

## 1. Verify the raw data in GCS

In [4]:
import pyarrow.fs as pafs

fs = pafs.GcsFileSystem()
info = fs.get_file_info(pafs.FileSelector(BUCKET, recursive=False))

paths = [f"{BUCKET}/{entry.base_name}" for entry in info]
paths

['your-bucket/wi-schools-raw.parquet']

## 2. Upload, compile, and inspect the Dataform pipeline
The three SQLX definitions are templated with the project, bucket, and dataset from `config.json`, written to the Dataform workspace, and compiled. The compiled action graph gives each table's dependencies.

In [6]:
# Dataform upload + compilation
from google.cloud import dataform

WORKSPACE = "dev"

dfm = dataform.DataformClient()

workspace_path = (
    f"projects/{PROJECT}/locations/{REGION}/repositories/{REPO}/workspaces/{WORKSPACE}"
)

# Upload each SQLX file
def upload_sqlx(filename):
    with open(f"definitions/{filename}", "r") as f:
        contents = f.read()
    contents = (contents.replace("__GCP_PROJECT__", PROJECT)
                        .replace("__GCS_BUCKET__", BUCKET)
                        .replace("__DATASET__", DATASET))

    req = dataform.WriteFileRequest(
        workspace=workspace_path,         
        path=f"definitions/{filename}",   
        contents=contents.encode("utf-8")
    )

    dfm.write_file(request=req)

# Upload all three definitions
upload_sqlx("wi_counties.sqlx")
upload_sqlx("schools.sqlx")
upload_sqlx("wi_county_schools.sqlx")

# Compile pipeline
compilation = dfm.create_compilation_result(
    request=dataform.CreateCompilationResultRequest(
        parent=f"projects/{PROJECT}/locations/{REGION}/repositories/{REPO}",
        compilation_result=dataform.CompilationResult(
            workspace=workspace_path
        ),
    )
)

assert compilation.compilation_errors == []
compilation_result_name = compilation.name
compilation_result_name

'projects/your-gcp-project/locations/us-central1/repositories/wi-schools-pipeline/compilationResults/a4d3a02c-a1f2-44e6-9290-c7dcece45b51'

In [7]:
response = dfm.query_compilation_result_actions(
    request={"name": compilation_result_name}
)

deps = {}

for action in response.compilation_result_actions:

    name = action.target.name

    if action.relation is not None:
        dep_names = [d.name for d in action.relation.dependency_targets]
    else:
        dep_names = []

    deps[name] = dep_names

deps

{'first_view': [],
 'schools': [],
 'second_view': ['first_view'],
 'wi_counties': [],
 'wi_county_schools': ['schools', 'wi_counties']}

## 3. Query the modeled tables in BigQuery

### Wisconsin counties loaded

In [12]:
from google.cloud import bigquery
bq = bigquery.Client()

query = f"""
SELECT COUNT(*) AS cnt
FROM `{PROJECT}.{DATASET}.wi_counties`
"""

df = bq.query(query).to_dataframe()
int(df['cnt'][0])

72

### Public schools matched to a county (`ST_CONTAINS`)

In [40]:
query = f"""
SELECT COUNT(*) AS cnt
FROM `{PROJECT}.{DATASET}.wi_county_schools`
WHERE agency_type = 'Public school'
"""
df = bq.query(query).to_dataframe()
int(df["cnt"][0])

2116

### Counties with at least 2 cities that have 3+ public high schools (pipe syntax)

In [39]:
pipe_query = f"""
FROM `{PROJECT}.{DATASET}.wi_county_schools`
|> WHERE agency_type = 'Public school'
|> WHERE school_type = 'High School'
|> AGGREGATE COUNT(*) AS hs_cnt GROUP BY county_name, city
|> WHERE hs_cnt >= 3
|> AGGREGATE COUNT(*) AS city_cnt GROUP BY county_name
|> WHERE city_cnt >= 2
|> SELECT county_name
|> ORDER BY county_name
"""
job = bq.query(pipe_query)
[county for county in job.to_dataframe()["county_name"]]

['Brown', 'Dane', 'Milwaukee', 'Waukesha']

### Cost check: how many runs of that query fit in 1 TiB scanned

In [35]:
bytes_per_run = job.total_bytes_processed
int((1024 ** 4) // bytes_per_run)

6797767

### Closest public high school to each Dane County middle school (`ST_DISTANCE`, `MIN_BY`)

In [38]:
closest_hs_query = f"""
WITH middle_schools AS (
  SELECT
    school_name AS middle_name,
    location AS middle_loc
  FROM `{PROJECT}.{DATASET}.wi_county_schools`
  WHERE county_name = 'Dane'
    AND agency_type = 'Public school'
    AND school_type = 'Middle School'
),
high_schools AS (
  SELECT
    school_name AS high_name,
    location AS high_loc
  FROM `{PROJECT}.{DATASET}.wi_county_schools`
  WHERE county_name = 'Dane'
    AND agency_type = 'Public school'
    AND school_type = 'High School'
)
SELECT
  m.middle_name,
  MIN_BY(h.high_name, ST_DISTANCE(m.middle_loc, h.high_loc)) AS closest_high
FROM middle_schools m
CROSS JOIN high_schools h
GROUP BY m.middle_name
ORDER BY m.middle_name
"""
df = bq.query(closest_hs_query).to_dataframe()
dict(zip(df["middle_name"], df["closest_high"]))

{'Badger Ridge Middle': 'Verona Area High',
 'Badger Rock Middle': 'West High',
 'Belleville Middle': 'Belleville High',
 'Black Hawk Middle': 'Shabazz-City High',
 'Central Heights Middle': 'Prairie Phoenix Academy',
 'Cherokee Heights Middle': 'Capital High',
 'De Forest Middle': 'De Forest High',
 'Deerfield Middle': 'Deerfield High',
 'Ezekiel Gillespie Middle School': 'Vel Phillips Memorial High School',
 'Glacial Drumlin School': 'LaFollette High',
 'Glacier Creek Middle': 'Middleton High',
 'Hamilton Middle': 'Capital High',
 'Indian Mound Middle': 'McFarland High',
 'Innovative and Alternative Middle': 'Innovative High',
 'James Wright Middle': 'West High',
 'Kromrey Middle': 'Middleton High',
 'Marshall Middle': 'Marshall High',
 'Mount Horeb Middle': 'Mount Horeb High',
 'Nikolay Middle': 'Koshkonong Trails School',
 "O'Keeffe Middle": 'Innovative High',
 'Oregon Middle': 'Oregon High',
 'Patrick Marsh Middle': 'Prairie Phoenix Academy',
 'Prairie View Middle': 'Sun Prairie W